# 📐 Math Intuition for GenAI
### Pre-work Notebook 2 of 3

**Time:** ~60 minutes &nbsp;|&nbsp; **Level:** No math background needed &nbsp;|&nbsp; **1 install required (NumPy)**

---

## Why this notebook?

Every word you've ever typed into ChatGPT gets converted into a **list of numbers** called a vector — before the model even reads it.

To understand GenAI at the M00 level, you need to know:
1. What is a vector?
2. How do you measure similarity between two vectors?
3. How does a model turn raw scores into probabilities?

No calculus. No matrices. Just intuition and simple arithmetic.

---

## Structure of this notebook

| Section | Topic | Time |
|---|---|---|
| 1 | Vectors — lists of numbers that carry meaning | 15 min |
| 2 | Dot product — measuring similarity | 15 min |
| 3 | Probability and softmax — how models make choices | 15 min |
| 4 | CHALLENGE — Word similarity search | 15 min (optional) |

In [ ]:
# Install NumPy — the math library used in almost every AI/ML project
# (already installed in Google Colab — this is just in case)
!pip install numpy -q

import numpy as np   # np is the standard short name everyone uses
print("NumPy version:", np.__version__)
print("Ready!")

---
## Section 1: Vectors — Numbers That Carry Meaning

### What is a vector?

**Analogy:** Imagine you want to describe a city using numbers:
- Population: 8.3 million
- Average temperature: 22°C
- Cost of living index: 85

You could write this as `[8300000, 22, 85]` — a list of numbers that **describes** that city. That's a vector.

In GenAI, words are described the same way — as lists of numbers. The model learns that `'king'` and `'queen'` have similar number lists, while `'king'` and `'pizza'` do not.

These number-lists are called **word embeddings**. You'll build them in M00.

In [ ]:
# ── WHAT IS A VECTOR? ─────────────────────────────────────────────────

# A Python list is a vector
city_london = [8300000, 22, 85]   # population, temp, cost
city_mumbai = [20000000, 30, 45]  # population, temp, cost

# NumPy makes math on lists fast and easy
# np.array() converts a Python list into a NumPy array (a proper vector)
london = np.array([8300000, 22, 85])
mumbai = np.array([20000000, 30, 45])

print("London vector:", london)
print("Mumbai vector:", mumbai)
print("Shape (how many numbers):", london.shape)  # → (3,) means 3 numbers

print()

# ── WORD VECTORS (simplified for intuition) ───────────────────────────
# Real word vectors have 300-1536 numbers
# Here we use 3 numbers to make it easy to understand:
# [royalty_score, gender_score, age_score]

king  = np.array([0.9, 0.1, 0.8])   # high royalty, male (low gender), adult
queen = np.array([0.9, 0.9, 0.8])   # high royalty, female (high gender), adult
boy   = np.array([0.1, 0.1, 0.0])   # no royalty, male, young (low age)
pizza = np.array([0.0, 0.5, 0.0])   # nothing in common with royalty

print("king  vector:", king)
print("queen vector:", queen)
print("boy   vector:", boy)
print("pizza vector:", pizza)
print()
print("Intuition: king and queen should be MORE similar than king and pizza.")
print("Next section: how do we actually measure that similarity?")

In [ ]:
# ✏️  YOUR TURN — Section 1

# Create simplified vectors for these words
# Use 3 numbers: [size_score, speed_score, danger_score]
# Scores between 0.0 (none) and 1.0 (maximum)

# Example: elephant → large, slow, mildly dangerous
elephant = np.array([0.9, 0.1, 0.4])

# Your turn — fill in the ___ with values that feel right
cheetah = np.array([___])   # small, very fast, somewhat dangerous
bee     = np.array([___])   # tiny, fast, dangerous (stings!)
lion    = np.array([___])   # large, fast, very dangerous

print("elephant:", elephant)
print("cheetah: ", cheetah)
print("bee:     ", bee)
print("lion:    ", lion)

# Think: which animal should be MOST similar to elephant?
# We'll find out in Section 2.

---
## Section 2: Dot Product — Measuring Similarity

### How do you know if two vectors are similar?

**Analogy:** Imagine two people each have a checklist of interests:
- Person A: loves hiking (9/10), loves cooking (7/10), hates gaming (1/10)
- Person B: loves hiking (8/10), loves cooking (6/10), hates gaming (2/10)

They're very similar! You can measure this by multiplying matching scores and summing them up. That's the **dot product**.

**Formula:** `dot(A, B) = A[0]×B[0] + A[1]×B[1] + A[2]×B[2] + ...`

- **High dot product** → vectors point in the same direction → words are similar
- **Low dot product** → vectors point in different directions → words are different

This is how GenAI finds relevant documents in RAG (Module 5) and how attention works in transformers (Module 0).

In [ ]:
# ── DOT PRODUCT FROM SCRATCH ──────────────────────────────────────────

# Let's do it manually first to understand what's happening
king  = np.array([0.9, 0.1, 0.8])   # [royalty, gender, age]
queen = np.array([0.9, 0.9, 0.8])
pizza = np.array([0.0, 0.5, 0.0])

# Manual dot product: multiply element by element, then sum
def dot_product_manual(a, b):
    total = 0
    for i in range(len(a)):       # go through each position
        total += a[i] * b[i]      # multiply matching elements, add to total
    return total

king_queen_score = dot_product_manual(king, queen)
king_pizza_score = dot_product_manual(king, pizza)

print(f"king · queen = {king_queen_score:.3f}")
print(f"king · pizza = {king_pizza_score:.3f}")
print(f"\nHigher score = more similar.")
print(f"king and queen are {'more' if king_queen_score > king_pizza_score else 'less'} similar than king and pizza.")

In [ ]:
# ── DOT PRODUCT WITH NUMPY (how it's done in real code) ───────────────

# np.dot() does the same thing — much faster for large vectors
king  = np.array([0.9, 0.1, 0.8])
queen = np.array([0.9, 0.9, 0.8])
boy   = np.array([0.1, 0.1, 0.0])
pizza = np.array([0.0, 0.5, 0.0])

print("Similarity scores with 'king':")
print(f"  king · queen = {np.dot(king, queen):.3f}")
print(f"  king · boy   = {np.dot(king, boy):.3f}")
print(f"  king · pizza = {np.dot(king, pizza):.3f}")

print()

# ── COSINE SIMILARITY (what production code uses) ─────────────────────
# Dot product alone can be misleading if vectors have different lengths
# Cosine similarity = dot product divided by the sizes of both vectors
# Result is between -1 and 1:
#   1.0  = identical direction (very similar)
#   0.0  = perpendicular (unrelated)
#  -1.0  = opposite direction (opposite meaning)

def cosine_similarity(a, b):
    dot = np.dot(a, b)                    # the dot product
    size_a = np.sqrt(np.dot(a, a))        # length (magnitude) of vector a
    size_b = np.sqrt(np.dot(b, b))        # length (magnitude) of vector b
    return dot / (size_a * size_b)        # normalise by sizes

print("Cosine similarity with 'king':")
print(f"  cosine(king, queen) = {cosine_similarity(king, queen):.3f}  ← most similar")
print(f"  cosine(king, boy)   = {cosine_similarity(king, boy):.3f}")
print(f"  cosine(king, pizza) = {cosine_similarity(king, pizza):.3f}  ← least similar")
print()
print("You'll use this exact function in M00 (Text to Numbers)!")

In [ ]:
# ✏️  YOUR TURN — Section 2
# Fill in the ___ blanks

# Use the animal vectors from Section 1
# (Pre-set defaults below — your own values from Section 1 will work too)
elephant = np.array([0.9, 0.1, 0.4])
cheetah  = np.array([0.3, 0.9, 0.5])   # small, fast, somewhat dangerous
bee      = np.array([0.0, 0.8, 0.7])   # tiny, fast, stings
lion     = np.array([0.8, 0.6, 0.9])   # large, fast, very dangerous

# Step 1: Calculate cosine similarity between elephant and each other animal
sim_cheetah = cosine_similarity(___, ___)
sim_bee     = cosine_similarity(___, ___)
sim_lion    = cosine_similarity(___, ___)

print("Similarity to 'elephant':")
print(f"  vs cheetah: {sim_cheetah:.3f}")
print(f"  vs bee:     {sim_bee:.3f}")
print(f"  vs lion:    {sim_lion:.3f}")

# Step 2: Find the most similar animal to elephant
scores = {'cheetah': sim_cheetah, 'bee': sim_bee, 'lion': sim_lion}
most_similar = max(scores, key=scores.get)   # finds the key with highest value
print(f"\nMost similar to elephant: {most_similar}")
# ✅ Expected: lion (both large animals with similar profiles)

---
## Section 3: Probability and Softmax

### How does a model decide what word comes next?

**The key insight from M00:** An LLM does ONE thing — predict the next token.

Given `"The cat sat on the ___"`, the model scores every word in its vocabulary:
- 'mat': score 8.5
- 'floor': score 6.2
- 'roof': score 2.1
- 'keyboard': score 0.3
- (50,000 other words with even lower scores)

But scores aren't probabilities — scores can be any number. The model needs to convert them into **probabilities that sum to 1** (so it can say: 73% chance of 'mat', 22% chance of 'floor', etc.).

That conversion is called **softmax**.

In [ ]:
# ── WHAT IS A PROBABILITY? ────────────────────────────────────────────

# A probability is a number between 0 and 1
# All probabilities in a group must sum to 1.0

# Example: weather tomorrow
weather_probs = {
    'sunny':  0.60,   # 60% chance
    'cloudy': 0.25,   # 25% chance
    'rainy':  0.15    # 15% chance
}

total = sum(weather_probs.values())
print(f"All probabilities sum to: {total}")   # → 1.0

print()

# A language model does the same thing — but for 50,000 words
# "Given 'The cat sat on the', what's the probability of each next word?"
next_word_probs = {
    'mat':      0.41,
    'floor':    0.28,
    'roof':     0.15,
    'table':    0.09,
    'keyboard': 0.04,
    # ... 49,995 more words with tiny probabilities
}

print("Most likely next words:")
for word, prob in sorted(next_word_probs.items(), key=lambda x: -x[1]):
    bar = '█' * int(prob * 40)   # visual bar proportional to probability
    print(f"  {word:<12} {bar} {prob:.0%}")

In [ ]:
# ── SOFTMAX — CONVERTING SCORES TO PROBABILITIES ──────────────────────

# The model outputs raw scores (called logits) — not probabilities
# Softmax converts them

# Step 1: Raw scores from the model for 5 candidate words
words  = ['mat',  'floor', 'roof', 'table', 'keyboard']
scores = np.array([8.5,    6.2,    2.1,    1.8,     0.3])

print("Raw scores (logits):", scores)

# Step 2: Softmax formula
# 1. Raise e (≈2.718) to the power of each score  →  makes all numbers positive
# 2. Divide each by the total                      →  makes them sum to 1

def softmax(scores):
    exp_scores = np.exp(scores)          # e^score for each score
    total = np.sum(exp_scores)           # sum of all e^scores
    probabilities = exp_scores / total   # divide each by total
    return probabilities

probs = softmax(scores)

print()
print("After softmax (probabilities):")
for word, score, prob in zip(words, scores, probs):
    bar = '█' * int(prob * 50)
    print(f"  {word:<12} score={score:4.1f}  prob={prob:.1%}  {bar}")

print(f"\nAll probabilities sum to: {probs.sum():.4f}")  # → 1.0000

print()
print("Notice: softmax AMPLIFIES the winner.")
print("'mat' had score 8.5 vs 6.2 for 'floor' (1.37x more)")
print(f"But after softmax: {probs[0]:.1%} vs {probs[1]:.1%} ({probs[0]/probs[1]:.1f}x more)")

In [ ]:
# ✏️  YOUR TURN — Section 3
# Fill in the ___ blanks

# The model is predicting the next word for: "I love eating ___"
# It has given these raw scores:

candidate_words = ['pizza', 'code', 'rocks', 'happiness', 'sleep']
raw_scores = np.array([7.2, 3.1, 1.5, 2.8, 5.9])

# Step 1: Apply softmax to get probabilities
probabilities = ___(raw_scores)

# Step 2: Print each word with its probability
print("Next word predictions for 'I love eating ___':")
for word, prob in zip(candidate_words, probabilities):
    print(f"  {word:<12} {prob:.1%}")

# Step 3: Find the most likely next word
best_index = np.argmax(___)           # argmax finds index of highest value
best_word = candidate_words[___]
print(f"\nModel picks: '{best_word}' with {probabilities[best_index]:.1%} probability")

# ✅ Expected: pizza should win with highest probability

---
## Section 4: CHALLENGE — Word Similarity Search

> **Optional — 15 minutes**

This challenge combines everything from this notebook:
- Vectors
- Cosine similarity
- Softmax

You'll build a tiny semantic search function — the same idea behind RAG (Module 5).

In [ ]:
# 🏆 CHALLENGE — Semantic Word Finder
# Given a query word, find the most similar word from a list

# Word vectors (simplified — 4 dimensions: royalty, power, gender, age)
word_vectors = {
    'king':     np.array([0.9, 0.8, 0.1, 0.8]),
    'queen':    np.array([0.9, 0.7, 0.9, 0.8]),
    'prince':   np.array([0.7, 0.4, 0.1, 0.3]),
    'princess': np.array([0.7, 0.3, 0.9, 0.3]),
    'dog':      np.array([0.0, 0.2, 0.5, 0.4]),
    'cat':      np.array([0.0, 0.1, 0.5, 0.3]),
    'pizza':    np.array([0.0, 0.0, 0.5, 0.0]),
    'castle':   np.array([0.8, 0.6, 0.5, 0.9]),
}

def cosine_similarity(a, b):
    """Returns similarity between two vectors (between 0 and 1)."""
    dot = np.dot(a, b)
    return dot / (np.linalg.norm(a) * np.linalg.norm(b))  # np.linalg.norm = vector length


def find_similar(query_word, word_vectors, top_n=3):
    """
    Finds the top_n most similar words to the query word.
    Returns a sorted list of (word, similarity_score) tuples.
    """
    # Step 1: Get the query word's vector
    query_vec = word_vectors[___]

    # Step 2: Calculate similarity of query with every other word
    similarities = {}
    for word, vec in word_vectors.items():
        if word == query_word:        # skip comparing the word with itself
            continue
        similarities[word] = cosine_similarity(___, ___)

    # Step 3: Sort by similarity (highest first) and return top_n
    sorted_words = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    return sorted_words[:___]


# Test your function
print("=== Most similar to 'king' ===")
for word, score in find_similar('king', word_vectors):
    print(f"  {word:<12} similarity = {score:.3f}")

print()
print("=== Most similar to 'cat' ===")
for word, score in find_similar('cat', word_vectors):
    print(f"  {word:<12} similarity = {score:.3f}")

print()
print("=== Most similar to 'castle' ===")
for word, score in find_similar('castle', word_vectors):
    print(f"  {word:<12} similarity = {score:.3f}")

---
## ✅ Notebook 2 Summary

| Concept | What it means | Where you'll see it |
|---|---|---|
| **Vector** | A list of numbers that represents a word/sentence | M00: Word2Vec, embeddings |
| **Dot product** | Multiply matching numbers, sum up → similarity score | M00: Attention, cosine similarity |
| **Cosine similarity** | Dot product normalised by vector sizes → -1 to 1 | M05: RAG retrieval |
| **Softmax** | Converts raw scores into probabilities that sum to 1 | M00: Next-token prediction |
| **argmax** | Find the index of the highest value | M00: Pick the most likely token |

---

**Next:** Open `03_neural_networks_intuition.ipynb` — you'll learn HOW a model gets its vectors in the first place (training!).

> 💡 **Connection to M00:** In M00, slide 4 shows exactly the softmax calculation you just built — applied to a vocabulary of 50,000 words. You now understand what that slide means.